In [11]:
import os
import glob
from typing import List, Dict, Tuple


def _sorted_nii(folder: str) -> List[str]:
    files = glob.glob(os.path.join(folder, "*.nii.gz")) + glob.glob(os.path.join(folder, "*.nii"))
    return sorted(files)


def _sid_from_path(p: str) -> str:
    return os.path.basename(p).replace(".nii.gz", "").replace(".nii", "")


def load_msd_dataset(task_root: str) -> Tuple[List[Dict], List[Dict]]:
    """
    MSD/nnUNet 风格数据集读取（安全配对版）
    返回:
      train_items: [{"image": path, "label": path, "id": str}, ...]
      test_items:  [{"image": path, "id": str}, ...]
    """
    imagesTr = os.path.join(task_root, "imagesTr")
    labelsTr = os.path.join(task_root, "labelsTr")
    imagesTs = os.path.join(task_root, "imagesTs")

    tr_images = _sorted_nii(imagesTr)
    tr_labels = _sorted_nii(labelsTr)

    if len(tr_images) == 0 or len(tr_labels) == 0:
        raise RuntimeError(f"Empty imagesTr/labelsTr under: {task_root}")
    if len(tr_images) != len(tr_labels):
        raise RuntimeError(f"Mismatch: imagesTr={len(tr_images)} labelsTr={len(tr_labels)}")

    # label 映射：id -> label_path
    label_map: Dict[str, str] = {}
    for lab in tr_labels:
        sid = _sid_from_path(lab)
        if sid in label_map:
            raise RuntimeError(f"Duplicate label id found: {sid}")
        label_map[sid] = lab

    # 用 image 的 id 去查 label，确保 1-1 对齐
    train_items: List[Dict] = []
    for img in tr_images:
        sid = _sid_from_path(img)
        if sid not in label_map:
            raise RuntimeError(f"Label missing for {sid}")
        train_items.append({"image": img, "label": label_map[sid], "id": sid})

    # 可选：也检查 label 是否存在对应 image（更严格）
    image_ids = {x["id"] for x in train_items}
    extra_labels = [sid for sid in label_map.keys() if sid not in image_ids]
    if len(extra_labels) > 0:
        raise RuntimeError(f"Labels without images: {extra_labels[:10]}")

    # test（可能不存在 labels）
    test_items: List[Dict] = []
    if os.path.isdir(imagesTs):
        ts_images = _sorted_nii(imagesTs)
        for img in ts_images:
            sid = _sid_from_path(img)
            test_items.append({"image": img, "id": sid})

    return train_items, test_items


def fixed_split(items: List[Dict], val_ratio: float = 0.2):
    """
    固定划分（可复现）：
    - 按 id 排序
    - 前 val_ratio 作为 val，其余 train
    """
    items = sorted(items, key=lambda x: x["id"])
    n = len(items)
    n_val = max(1, int(round(n * val_ratio)))
    val_items = items[:n_val]
    train_items = items[n_val:]
    return train_items, val_items


def assert_no_overlap(tr: List[Dict], va: List[Dict]):
    tr_ids = {x["id"] for x in tr}
    va_ids = {x["id"] for x in va}
    inter = tr_ids & va_ids
    if len(inter) != 0:
        raise RuntimeError(f"Train/Val overlap: {sorted(list(inter))[:10]}")

In [12]:
train_items, test_items = load_msd_dataset("/home/pumengyu/First2TB/PuMengYu/CT/segmentation/Task02_Heart")
tr, va = fixed_split(train_items, 0.2)
assert_no_overlap(tr, va)
print(len(train_items), len(tr), len(va), len(test_items))
print(train_items[0])

20 16 4 10
{'image': '/home/pumengyu/First2TB/PuMengYu/CT/segmentation/Task02_Heart/imagesTr/la_003.nii.gz', 'label': '/home/pumengyu/First2TB/PuMengYu/CT/segmentation/Task02_Heart/labelsTr/la_003.nii.gz', 'id': 'la_003'}


In [13]:
tr_images = [
    "imagesTr/la_001.nii.gz",
    "imagesTr/la_002.nii.gz",
]

tr_labels = [
    "labelsTr/la_001.nii.gz",
    "labelsTr/la_002.nii.gz",
]

def _sid_from_path(p):
    return p.split("/")[-1].replace(".nii.gz", "")

# 构造 label_map
label_map = {}
for lab in tr_labels:
    sid = _sid_from_path(lab)
    label_map[sid] = lab

print("label_map:")
print(label_map)

# 构造 train_items
train_items = []
for img in tr_images:
    sid = _sid_from_path(img)
    train_items.append({
        "image": img,
        "label": label_map[sid],
        "id": sid
    })

print("\ntrain_items:")
for item in train_items:
    print(item)

label_map:
{'la_001': 'labelsTr/la_001.nii.gz', 'la_002': 'labelsTr/la_002.nii.gz'}

train_items:
{'image': 'imagesTr/la_001.nii.gz', 'label': 'labelsTr/la_001.nii.gz', 'id': 'la_001'}
{'image': 'imagesTr/la_002.nii.gz', 'label': 'labelsTr/la_002.nii.gz', 'id': 'la_002'}
